In [2]:
import os
import numpy as np
import torch


EMBED_PATH     = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/模型文件/embeddings_best.pt"
BASIS_DIR      = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/pattern_basis"
INFLUENCE_PATH = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/pattern_influence/influence_scores.pt"
SAVE_DIR = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/遗忘实验"
os.makedirs(SAVE_DIR, exist_ok=True)

TOP_K           = 8
LAMBDA_SCHEDULE = [1.0, 0.9, 0.8, 0.6, 0.4]

# ── 1. 加载数据 ────────────────────────────────────────────────────────────────
print("[1] 加载数据...")
ckpt     = torch.load(EMBED_PATH, map_location="cpu")
user_emb = ckpt["user_embeddings"].numpy()
item_emb = ckpt["item_embeddings"].numpy()

basis_ckpt    = torch.load(f"{BASIS_DIR}/pattern_basis.pt", map_location="cpu")
pattern_basis = basis_ckpt["pattern_basis"].numpy()

infl_ckpt        = torch.load(INFLUENCE_PATH, map_location="cpu", weights_only=False)
influence_matrix = infl_ckpt["influence_matrix"]   # (n_sampled, K)
sampled_uids     = infl_ckpt["sampled_uids"]
hq_patterns      = infl_ckpt["hq_patterns"]
n_sampled        = len(sampled_uids)
K                = len(hq_patterns)
print(f"  用户数: {n_sampled}, 高质量 pattern 数: {K}")

# ── 2. 逐用户自适应阈值：μ_u + σ_u ───────────────────────────────────────────
print("\n[2] 计算逐用户自适应阈值 τ_u = μ_u + σ_u ...")
mu_u    = influence_matrix.mean(axis=1)    # (n_sampled,)
sigma_u = influence_matrix.std(axis=1)    # (n_sampled,)
tau_u   = mu_u + sigma_u                  # (n_sampled,)

print(f"  τ_u 统计: mean={tau_u.mean():.4f}, std={tau_u.std():.4f}, "
      f"min={tau_u.min():.4f}, max={tau_u.max():.4f}")

n_forget_arr = np.array([
    int((influence_matrix[idx] > tau_u[idx]).sum())
    for idx in range(n_sampled)
])
print(f"  每用户遗忘 pattern 数: mean={n_forget_arr.mean():.2f}, "
      f"min={n_forget_arr.min()}, max={n_forget_arr.max()}")
print(f"  遗忘 0 个 pattern 的用户数: {(n_forget_arr == 0).sum()}")

# ── 3. Projection-Based Unlearning ───────────────────────────────────────────
print("\n[3] 执行 Projection-Based Unlearning...")
user_emb_before = user_emb.copy()
user_emb_after  = user_emb.copy()

for idx, uid in enumerate(sampled_uids):
    scores    = influence_matrix[idx]
    threshold = tau_u[idx]
    sorted_ki = np.argsort(scores)[::-1]
    targets   = [ki for ki in sorted_ki if scores[ki] > threshold]

    if len(targets) == 0:
        continue

    eu = user_emb_after[uid].copy()
    for t, ki in enumerate(targets):
        lam = LAMBDA_SCHEDULE[t] if t < len(LAMBDA_SCHEDULE) else LAMBDA_SCHEDULE[-1]
        vk  = pattern_basis[hq_patterns[ki]]
        eu  = eu - lam * np.dot(eu, vk) * vk
    user_emb_after[uid] = eu

print("  Unlearning 完成")

# ── 4. 评估：PFR 与 Overlap@K ─────────────────────────────────────────────────
print("\n[4] 评估 PFR 与 Overlap@K...")

pfr_list     = []
overlap_list = []

for idx, uid in enumerate(sampled_uids):
    scores    = influence_matrix[idx]
    threshold = tau_u[idx]
    sorted_ki = np.argsort(scores)[::-1]
    targets   = [ki for ki in sorted_ki if scores[ki] > threshold]

    if len(targets) == 0:
        continue

    eu_b = user_emb_before[uid]
    eu_a = user_emb_after[uid]

    # PFR
    A_before = sum(abs(np.dot(eu_b, pattern_basis[hq_patterns[ki]])) for ki in targets)
    A_after  = sum(abs(np.dot(eu_a, pattern_basis[hq_patterns[ki]])) for ki in targets)
    pfr = 1.0 - A_after / (A_before + 1e-10) if A_before > 0 else 0.0
    pfr_list.append(pfr)

    # Overlap@K
    topk_b = set(np.argpartition(item_emb @ eu_b, -TOP_K)[-TOP_K:])
    topk_a = set(np.argpartition(item_emb @ eu_a, -TOP_K)[-TOP_K:])
    overlap_list.append(len(topk_b & topk_a) / TOP_K)

print(f"  有效遗忘用户数: {len(pfr_list)}")
print(f"  PFR        — mean={np.mean(pfr_list):.4f}, std={np.std(pfr_list):.4f}, "
      f"min={np.min(pfr_list):.4f}, max={np.max(pfr_list):.4f}")
print(f"  Overlap@{TOP_K} — mean={np.mean(overlap_list):.4f}, std={np.std(overlap_list):.4f}, "
      f"min={np.min(overlap_list):.4f}, max={np.max(overlap_list):.4f}")

# ── 5. 前10名有效遗忘用户示例 ─────────────────────────────────────────────────
print("\n[5] 前10名有效遗忘用户详情：")
shown = 0
for idx, uid in enumerate(sampled_uids):
    if shown >= 10:
        break
    scores    = influence_matrix[idx]
    threshold = tau_u[idx]
    sorted_ki = np.argsort(scores)[::-1]
    targets   = [ki for ki in sorted_ki if scores[ki] > threshold]
    if len(targets) == 0:
        continue

    forgotten = [(hq_patterns[ki], round(float(scores[ki]), 4)) for ki in targets]
    eu_b = user_emb_before[uid]
    eu_a = user_emb_after[uid]

    topk_b  = set(np.argpartition(item_emb @ eu_b, -TOP_K)[-TOP_K:])
    topk_a  = set(np.argpartition(item_emb @ eu_a, -TOP_K)[-TOP_K:])
    overlap = len(topk_b & topk_a) / TOP_K

    A_before = sum(abs(np.dot(eu_b, pattern_basis[hq_patterns[ki]])) for ki in targets)
    A_after  = sum(abs(np.dot(eu_a, pattern_basis[hq_patterns[ki]])) for ki in targets)
    pfr_u   = 1.0 - A_after / (A_before + 1e-10) if A_before > 0 else 0.0

    print(f"  uid={uid} | τ_u={threshold:.4f} | 遗忘={forgotten} | "
          f"PFR={pfr_u:.4f} | Overlap@{TOP_K}={overlap:.4f}")
    shown += 1

[1] 加载数据...
  用户数: 1000, 高质量 pattern 数: 8

[2] 计算逐用户自适应阈值 τ_u = μ_u + σ_u ...
  τ_u 统计: mean=0.4147, std=0.0804, min=0.2344, max=0.6519
  每用户遗忘 pattern 数: mean=1.48, min=0, max=3
  遗忘 0 个 pattern 的用户数: 11

[3] 执行 Projection-Based Unlearning...
  Unlearning 完成

[4] 评估 PFR 与 Overlap@K...
  有效遗忘用户数: 989
  PFR        — mean=0.9775, std=0.0266, min=0.8746, max=1.0000
  Overlap@8 — mean=0.5880, std=0.2572, min=0.0000, max=1.0000

[5] 前10名有效遗忘用户详情：
  uid=13738 | τ_u=0.5026 | 遗忘=[(5, 0.6517)] | PFR=1.0000 | Overlap@8=0.3750
  uid=25771 | τ_u=0.3310 | 遗忘=[(5, 0.3568), (4, 0.3447), (1, 0.3343)] | PFR=0.8990 | Overlap@8=0.8750
  uid=9664 | τ_u=0.3481 | 遗忘=[(4, 0.4351), (1, 0.3583)] | PFR=0.9594 | Overlap@8=0.7500
  uid=14105 | τ_u=0.4262 | 遗忘=[(2, 0.5033)] | PFR=1.0000 | Overlap@8=0.6250
  uid=16776 | τ_u=0.5280 | 遗忘=[(4, 0.7309)] | PFR=1.0000 | Overlap@8=0.2500
  uid=14194 | τ_u=0.4240 | 遗忘=[(4, 0.5108)] | PFR=1.0000 | Overlap@8=0.6250
  uid=11562 | τ_u=0.5098 | 遗忘=[(1, 0.6492)] | PFR=1.0000 | O

In [6]:
# ── 以下变量假设遗忘实验已在同一 session 中运行完毕 ──────────────────────────
# 直接复用内存中已有的：
#   sampled_uids, hq_patterns, influence_matrix
#   mu_u, sigma_u, tau_u, n_forget_arr
#   user_emb_before, user_emb_after
#   pfr_list, overlap_list

# 1. 遗忘前后 embedding
torch.save(
    {"user_emb_before": user_emb_before,
     "user_emb_after":  user_emb_after,
     "sampled_uids":    sampled_uids},
    os.path.join(SAVE_DIR, "embeddings_unlearned.pt")
)

# 2. 阈值与遗忘 pattern 数
torch.save(
    {"mu_u":         mu_u,
     "sigma_u":      sigma_u,
     "tau_u":        tau_u,
     "n_forget_arr": n_forget_arr,
     "hq_patterns":  hq_patterns},
    os.path.join(SAVE_DIR, "threshold_info.pt")
)

# 3. 评估指标
torch.save(
    {"pfr_list":     pfr_list,
     "overlap_list": overlap_list,
     "pfr_mean":     float(np.mean(pfr_list)),
     "pfr_std":      float(np.std(pfr_list)),
     "pfr_min":      float(np.min(pfr_list)),
     "pfr_max":      float(np.max(pfr_list)),
     "overlap_mean": float(np.mean(overlap_list)),
     "overlap_std":  float(np.std(overlap_list)),
     "overlap_min":  float(np.min(overlap_list)),
     "overlap_max":  float(np.max(overlap_list)),
     "n_valid_users": len(pfr_list),
     "top_k":        8,
     "lambda_schedule": [1.0, 0.9, 0.8, 0.6, 0.4]},
    os.path.join(SAVE_DIR, "eval_results.pt")
)

print(f"[完成] 保存至 {SAVE_DIR}")
print(f"  embeddings_unlearned.pt — 遗忘前后 user embedding")
print(f"  threshold_info.pt       — τ_u / μ_u / σ_u / 每用户遗忘 pattern 数")
print(f"  eval_results.pt         — PFR & Overlap@8 完整统计")
print(f"\n汇总：")
print(f"  有效遗忘用户数: {len(pfr_list)}")
print(f"  PFR        — mean={np.mean(pfr_list):.4f}, std={np.std(pfr_list):.4f}")
print(f"  Overlap@8  — mean={np.mean(overlap_list):.4f}, std={np.std(overlap_list):.4f}")

[完成] 保存至 /Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/遗忘实验
  embeddings_unlearned.pt — 遗忘前后 user embedding
  threshold_info.pt       — τ_u / μ_u / σ_u / 每用户遗忘 pattern 数
  eval_results.pt         — PFR & Overlap@8 完整统计

汇总：
  有效遗忘用户数: 989
  PFR        — mean=0.9775, std=0.0266
  Overlap@8  — mean=0.5880, std=0.2572
